In [9]:
import os
import pandas as pd
import numpy as np 
import seaborn as sns
from tqdm import tqdm

from pydub import AudioSegment
from pydub.utils import make_chunks
from IPython.display import Audio

import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [20]:
# creating the dataframe which has path to wav file and it's label

df = pd.read_csv('./data/data_prueba_preprocessed.csv')
df = df[['audio_path', 'label']]
print(df.shape)

(1917, 2)


In [30]:
df.sample(3)

,audio_path,label
1336,./transfor_songs/converted_audio_files/274503.mp3,1
206,./transfor_songs/converted_audio_files/316755.mp3,1
1764,./transfor_songs/converted_audio_files/249791.mp3,0


In [22]:
def get_split_unique(path, base_filename):
    """
    Splits an audio file into 1-second chunks and names each chunk uniquely.

    Args:
        path (str): The file path to the input audio file.
        base_filename (str): The base name to use for naming the chunk files.

    Returns:
        list: A list of file paths to the exported audio chunks.
    """
    myaudio = AudioSegment.from_file(path, "mp3")
    chunk_length_ms = 1500  # Size of each chunk (1 second in this example)
    chunks = make_chunks(myaudio, chunk_length_ms)

    # Ensure that the chunks directory exists
    if not os.path.exists("chunks"):
        os.makedirs("chunks")

    paths = []
    for i, chunk in enumerate(chunks):
        chunk_name = f"chunks/{base_filename}_chunk{i}.mp3"  # Unique naming
        paths.append(chunk_name)
        chunk.export(chunk_name, format="mp3")

    return paths

def get_split_with_label(row):
    """
    Splits an audio file into chunks and associates each chunk with the audio's original label.

    Args:
        row (pd.Series): A row from a DataFrame containing 'audio_path' and 'label'.

    Returns:
        list of tuples: A list of tuples, where each tuple contains the path to a chunk and the label.
    """
    path = row['audio_path']
    label = row['label']
    base_filename = os.path.splitext(os.path.basename(path))[0]  # Use the base file name
    chunk_paths = get_split_unique(path, base_filename)
    return [(chunk_path, label) for chunk_path in chunk_paths]

In [ ]:
chunk_data = df.apply(get_split_with_label, axis=1)
chunk_data = [item for sublist in chunk_data for item in sublist]
chunk_df = pd.DataFrame(chunk_data, columns=['audio_path', 'label'])

In [35]:
print(chunk_df.shape)
chunk_df.sample(4)

(60674, 2)


,audio_path,label
33793,chunks/925328_chunk41.mp3,1
55793,chunks/892772_chunk26.mp3,0
13301,chunks/244886_chunk67.mp3,1
18378,chunks/292264_chunk0.mp3,1
